# Systolic AXI-Lite Compute Core Correctness Test

This notebook tests `SystolicAxiLiteWrapper` through Python MMIO.

Flow:
1. Load the bitstream / overlay.
2. Find the `SystolicAxiLiteWrapper` AXI-Lite base address.
3. Load `hex/case*/act.hex`, `weight.hex`, `bias.hex`, and `golden.hex`.
4. Write activation/bias/weight through `InputLoadFSM`.
5. Start the systolic core.
6. Read 256 output words from the output window and compare with golden.

This is a compute core correctness test. It does not measure final DDR/data-mover performance.

In [1]:
from pathlib import Path
import time

from pynq import Overlay, MMIO

NOTEBOOK_DIR = Path.cwd()
HEX_ROOT = NOTEBOOK_DIR / "hex"

# If None, the notebook auto-selects the only .bit file in this folder.
# If there are multiple .bit files, set this explicitly, for example:
# BITSTREAM_PATH = NOTEBOOK_DIR / "design_1_wrapper.bit"
BITSTREAM_PATH = None

# If Overlay.ip_dict cannot find the custom IP, set this manually from Vivado Address Editor.
# Example: MANUAL_BASE_ADDR = 0x43C00000
MANUAL_BASE_ADDR = None

MMIO_RANGE = 0x1000
LOAD_OVERLAY = True

def resolve_bitstream_path():
    if BITSTREAM_PATH is not None:
        path = Path(BITSTREAM_PATH)
        if not path.exists():
            raise FileNotFoundError(f"Bitstream not found: {path}")
        return path

    candidates = sorted(NOTEBOOK_DIR.glob("*.bit"))
    if len(candidates) == 1:
        return candidates[0]
    if len(candidates) == 0:
        raise FileNotFoundError(
            "No .bit file found in this notebook folder. Copy your .bit/.hwh here or set BITSTREAM_PATH."
        )
    raise RuntimeError(
        "Multiple .bit files found. Set BITSTREAM_PATH explicitly: "
        + ", ".join(str(p.name) for p in candidates)
    )

if LOAD_OVERLAY:
    bitstream_path = resolve_bitstream_path()
    overlay = Overlay(str(bitstream_path))
    overlay.download()
    print(f"Loaded overlay: {bitstream_path}")
    print("Available IP names:")
    for name in overlay.ip_dict.keys():
        print("  ", name)
else:
    overlay = None
    print("Overlay loading skipped. Make sure the bitstream is already programmed and MANUAL_BASE_ADDR is set.")

Loaded overlay: /home/xilinx/jupyter_notebooks/FPGA_AOC_systolic/design_1_wrapper.bit
Available IP names:
   SystolicAxiLiteWrapp_0/s_axi
   processing_system7_0


In [2]:
# SystolicAxiLiteWrapper register map
REG_CONTROL = 0x000
REG_K_TILE_CNT = 0x004
REG_ACT_BASE = 0x008
REG_W_BASE = 0x00C
REG_BIAS_BASE = 0x010
REG_STATUS = 0x014
REG_INPUT_TARGET = 0x018
REG_INPUT_COUNT = 0x01C
REG_OUTPUT_COUNT = 0x020

# Hardware profiling counters. These are measured by RTL, not estimated in Python.
REG_TOTAL_CYCLES = 0x024
REG_COMPUTE_BUSY_CYCLES = 0x028
REG_WEIGHT_LOAD_CYCLES = 0x02C
REG_OVERLAP_CYCLES = 0x030
REG_WEIGHT_STALL_CYCLES = 0x034
REG_SYSTOLIC_LOAD_CYCLES = 0x038
REG_WEIGHT_WORD_ACCEPTS = 0x03C
REG_ACT_BRAM_READS = 0x040
REG_WEIGHT_BRAM_READS = 0x044
REG_BIAS_BRAM_READS = 0x048
REG_OUTPUT_WORD_WRITES = 0x04C
REG_INPUT_BRAM_ACCESS_CYCLES = 0x050
REG_CORE_BRAM_ACCESS_CYCLES = 0x054

PROFILE_COUNTER_REGS = {
    "total_cycle_count": REG_TOTAL_CYCLES,
    "compute_busy_cycles": REG_COMPUTE_BUSY_CYCLES,
    "weight_load_cycles": REG_WEIGHT_LOAD_CYCLES,
    "overlap_cycles": REG_OVERLAP_CYCLES,
    "stall_cycles_due_to_w_bram_valid_0": REG_WEIGHT_STALL_CYCLES,
    "systolic_load_cycles": REG_SYSTOLIC_LOAD_CYCLES,
    "weight_word_accepts": REG_WEIGHT_WORD_ACCEPTS,
    "activation_bram_reads": REG_ACT_BRAM_READS,
    "weight_bram_reads": REG_WEIGHT_BRAM_READS,
    "bias_bram_reads": REG_BIAS_BRAM_READS,
    "output_word_writes": REG_OUTPUT_WORD_WRITES,
    "input_bram_access_cycles": REG_INPUT_BRAM_ACCESS_CYCLES,
    "core_bram_access_cycles": REG_CORE_BRAM_ACCESS_CYCLES,
}

INPUT_CTRL = 0x100
INPUT_BASE = 0x104
INPUT_COUNT = 0x108
INPUT_DATA = 0x10C

OUTPUT_BASE = 0x400

TARGET_ACT = 0
TARGET_BIAS = 1
TARGET_WEIGHT = 2

STATUS_MODULE_READY = 1 << 0
STATUS_INPUT_BUSY = 1 << 1
STATUS_INPUT_DONE = 1 << 2
STATUS_INPUT_ERROR = 1 << 3
STATUS_OUTPUT_BUSY = 1 << 4
STATUS_OUTPUT_DONE = 1 << 5
STATUS_OUTPUT_OVERFLOW = 1 << 6
STATUS_WEIGHT_ACTIVE_VALID = 1 << 7
STATUS_WEIGHT_WRITE_FULL = 1 << 8
STATUS_WEIGHT_SWAP_DONE = 1 << 9
STATUS_WEIGHT_SYSTOLIC_ADDR_ERROR = 1 << 10
STATUS_WEIGHT_LOADER_ADDR_ERROR = 1 << 11

ERROR_STATUS_MASK = (
    STATUS_INPUT_ERROR
    | STATUS_OUTPUT_OVERFLOW
    | STATUS_WEIGHT_SYSTOLIC_ADDR_ERROR
    | STATUS_WEIGHT_LOADER_ADDR_ERROR
)

def find_systolic_ip(overlay):
    if overlay is None:
        raise RuntimeError("Overlay is not loaded. Set MANUAL_BASE_ADDR or LOAD_OVERLAY=True.")

    keywords = ("systolic", "SystolicAxiLiteWrapper", "SystolicAxiLiteWrapp")
    matches = []
    for name, info in overlay.ip_dict.items():
        lname = name.lower()
        if any(k.lower() in lname for k in keywords):
            matches.append((name, info))

    if not matches:
        raise RuntimeError(
            "Cannot find SystolicAxiLiteWrapper in overlay.ip_dict. "
            "Set MANUAL_BASE_ADDR from Vivado Address Editor."
        )

    name, info = matches[0]
    base_addr = int(info["phys_addr"])
    addr_range = int(info.get("addr_range", MMIO_RANGE))
    return name, base_addr, addr_range

if MANUAL_BASE_ADDR is not None:
    ip_name = "manual"
    base_addr = int(MANUAL_BASE_ADDR)
    addr_range = MMIO_RANGE
else:
    ip_name, base_addr, addr_range = find_systolic_ip(overlay)

mmio = MMIO(base_addr, max(MMIO_RANGE, addr_range))
print(f"Using IP: {ip_name}")
print(f"Base address: 0x{base_addr:08X}, range: 0x{max(MMIO_RANGE, addr_range):X}")

Using IP: SystolicAxiLiteWrapp_0/s_axi
Base address: 0x43C00000, range: 0x1000


In [3]:
BIAS_WORDS = 16
GOLDEN_WORDS = 256
TILE_WORDS = 64

CASE_CONFIG = {
    1: {"k_cnt": 1, "data_words": 64, "total_tiles": 1},
    2: {"k_cnt": 2, "data_words": 128, "total_tiles": 2},
    3: {"k_cnt": 4, "data_words": 256, "total_tiles": 4},
    4: {"k_cnt": 96, "data_words": 6144, "total_tiles": 96},
}

def u32(value):
    return int(value) & 0xFFFFFFFF

def s32(value):
    value = int(value) & 0xFFFFFFFF
    return value - 0x100000000 if value & 0x80000000 else value

def read_hex_words(path):
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(path)

    words = []
    for raw in path.read_text().splitlines():
        line = raw.strip()
        if not line or line.startswith("#") or line.startswith("//"):
            continue
        token = line.split()[0]
        words.append(int(token, 16) & 0xFFFFFFFF)
    return words

def load_case(case_id):
    if case_id not in CASE_CONFIG:
        raise ValueError(f"Unsupported case_id={case_id}. Use one of {sorted(CASE_CONFIG)}")

    case_dir = HEX_ROOT / f"case{case_id}"
    cfg = dict(CASE_CONFIG[case_id])
    cfg["case_id"] = case_id
    cfg["act"] = read_hex_words(case_dir / "act.hex")
    cfg["weight"] = read_hex_words(case_dir / "weight.hex")
    cfg["bias"] = read_hex_words(case_dir / "bias.hex")
    cfg["golden"] = read_hex_words(case_dir / "golden.hex")

    assert len(cfg["act"]) >= cfg["data_words"], "act.hex is shorter than expected"
    assert len(cfg["weight"]) >= cfg["data_words"], "weight.hex is shorter than expected"
    assert len(cfg["bias"]) >= BIAS_WORDS, "bias.hex is shorter than expected"
    assert len(cfg["golden"]) >= GOLDEN_WORDS, "golden.hex is shorter than expected"
    return cfg

print(f"HEX_ROOT = {HEX_ROOT}")
print("Available cases:", sorted(CASE_CONFIG))

HEX_ROOT = /home/xilinx/jupyter_notebooks/FPGA_AOC_systolic/hex
Available cases: [1, 2, 3, 4]


In [4]:
def write_reg(offset, value):
    mmio.write(offset, u32(value))

def read_reg(offset):
    return u32(mmio.read(offset))

def read_status():
    return read_reg(REG_STATUS)

def format_status(status):
    names = [
        (STATUS_MODULE_READY, "module_ready"),
        (STATUS_INPUT_BUSY, "input_busy"),
        (STATUS_INPUT_DONE, "input_done"),
        (STATUS_INPUT_ERROR, "input_error"),
        (STATUS_OUTPUT_BUSY, "output_busy"),
        (STATUS_OUTPUT_DONE, "output_done"),
        (STATUS_OUTPUT_OVERFLOW, "output_overflow"),
        (STATUS_WEIGHT_ACTIVE_VALID, "weight_active_valid"),
        (STATUS_WEIGHT_WRITE_FULL, "weight_write_full"),
        (STATUS_WEIGHT_SWAP_DONE, "weight_swap_done"),
        (STATUS_WEIGHT_SYSTOLIC_ADDR_ERROR, "weight_systolic_addr_error"),
        (STATUS_WEIGHT_LOADER_ADDR_ERROR, "weight_loader_addr_error"),
    ]
    active = [name for mask, name in names if status & mask]
    return f"0x{status:08X}" + (" [" + ", ".join(active) + "]" if active else "")

def wait_for_status(mask, timeout_s=10.0, poll_interval_s=0.001, label="status"):
    t0 = time.time()
    last = read_status()
    while (last & mask) != mask:
        if time.time() - t0 > timeout_s:
            raise TimeoutError(f"Timeout waiting for {label}; last status {format_status(last)}")
        time.sleep(poll_interval_s)
        last = read_status()
    return last

print("Initial status:", format_status(read_status()))

Initial status: 0x00000001 [module_ready]


In [5]:
def input_loader_start(target, base_addr, words):
    write_reg(INPUT_BASE, base_addr)
    write_reg(INPUT_COUNT, words)
    write_reg(INPUT_CTRL, (target << 1) | 0x1)

def check_input_done(timeout_s=10.0):
    status = wait_for_status(STATUS_INPUT_DONE, timeout_s=timeout_s, label="input_done")
    if status & STATUS_INPUT_ERROR:
        raise RuntimeError(f"InputLoadFSM error: {format_status(status)}")
    return status

def load_input_words(target, words, base_addr=0, label="input", verbose=True):
    input_loader_start(target, base_addr, len(words))
    for idx, word in enumerate(words):
        write_reg(INPUT_DATA, word)
        if verbose and len(words) >= 1024 and (idx + 1) % 1024 == 0:
            print(f"  {label}: wrote {idx + 1}/{len(words)} words")
    status = check_input_done(timeout_s=max(10.0, len(words) * 0.01))
    if verbose:
        print(f"  {label}: done, {format_status(status)}")
    return status

def load_activation(words, verbose=True):
    return load_input_words(TARGET_ACT, words, base_addr=0, label="activation", verbose=verbose)

def load_bias(words, verbose=True):
    return load_input_words(TARGET_BIAS, words, base_addr=0, label="bias", verbose=verbose)

def load_weight_tile(weight_words, tile_idx, verbose=True):
    start = tile_idx * TILE_WORDS
    stop = start + TILE_WORDS
    tile_words = weight_words[start:stop]
    if len(tile_words) != TILE_WORDS:
        raise ValueError(f"Weight tile {tile_idx} has {len(tile_words)} words, expected {TILE_WORDS}")
    return load_input_words(TARGET_WEIGHT, tile_words, base_addr=0, label=f"weight tile {tile_idx}", verbose=verbose)

In [6]:
def reload_design():
    global overlay, mmio, ip_name, base_addr, addr_range
    if not LOAD_OVERLAY:
        print("LOAD_OVERLAY=False; not reloading design. Make sure hardware state is clean.")
        return
    overlay.download()
    if MANUAL_BASE_ADDR is None:
        ip_name, base_addr, addr_range = find_systolic_ip(overlay)
    else:
        ip_name, base_addr, addr_range = "manual", int(MANUAL_BASE_ADDR), MMIO_RANGE
    mmio = MMIO(base_addr, max(MMIO_RANGE, addr_range))

def start_systolic(k_cnt, act_base=0, weight_base=0, bias_base=0):
    write_reg(REG_K_TILE_CNT, k_cnt)
    write_reg(REG_ACT_BASE, act_base)
    write_reg(REG_W_BASE, weight_base)
    write_reg(REG_BIAS_BASE, bias_base)
    write_reg(REG_CONTROL, 0x1)

def read_outputs():
    return [read_reg(OUTPUT_BASE + i * 4) for i in range(GOLDEN_WORDS)]

def read_profile_counters():
    return {name: read_reg(offset) for name, offset in PROFILE_COUNTER_REGS.items()}

def summarize_pingpong(profile):
    total = int(profile["total_cycle_count"])
    overlap = int(profile["overlap_cycles"])
    no_overlap_total_est = total + overlap
    speedup = (no_overlap_total_est / total) if total else 0.0
    return {
        "no_overlap_total_cycles_est": no_overlap_total_est,
        "pingpong_saved_cycles_actual": overlap,
        "pingpong_speedup_vs_no_overlap_est": speedup,
    }

def print_profile_counters(profile):
    print("  hardware counters:")
    for name in PROFILE_COUNTER_REGS:
        print(f"    {name}: {profile[name]}")
    pingpong = summarize_pingpong(profile)
    print(
        "    no_overlap_total_cycles_est: "
        f"{pingpong['no_overlap_total_cycles_est']} "
        f"(actual total + actual overlap)"
    )
    print(
        "    pingpong_speedup_vs_no_overlap_est: "
        f"{pingpong['pingpong_speedup_vs_no_overlap_est']:.4f}x"
    )

def compare_outputs(got, golden, max_print=16):
    mismatches = []
    for idx, (g, ref) in enumerate(zip(got, golden)):
        if u32(g) != u32(ref):
            mismatches.append((idx, u32(g), u32(ref)))

    for idx, got_word, ref_word in mismatches[:max_print]:
        print(
            f"  mismatch[{idx:03d}]: got 0x{got_word:08X} ({s32(got_word)}), "
            f"golden 0x{ref_word:08X} ({s32(ref_word)})"
        )
    if len(mismatches) > max_print:
        print(f"  ... {len(mismatches) - max_print} more mismatches")
    return mismatches

def run_case(case_id, reload_before=True, verbose=True):
    cfg = load_case(case_id)
    if reload_before:
        reload_design()

    print(f"\n=== Running case{case_id} ===")
    print(
        f"k_cnt={cfg['k_cnt']}, data_words={cfg['data_words']}, "
        f"total_tiles={cfg['total_tiles']}"
    )

    t0 = time.time()
    load_activation(cfg["act"][:cfg["data_words"]], verbose=verbose)
    load_bias(cfg["bias"][:BIAS_WORDS], verbose=verbose)

    load_weight_tile(cfg["weight"], 0, verbose=verbose)
    wait_for_status(STATUS_WEIGHT_ACTIVE_VALID, timeout_s=10.0, label="weight_active_valid")

    start_systolic(cfg["k_cnt"], act_base=0, weight_base=0, bias_base=0)

    for tile_idx in range(1, cfg["total_tiles"]):
        load_weight_tile(cfg["weight"], tile_idx, verbose=verbose)

    status = wait_for_status(
        STATUS_OUTPUT_DONE,
        timeout_s=max(10.0, cfg["total_tiles"] * 2.0),
        label="output_done",
    )
    print("  final status:", format_status(status))
    if status & ERROR_STATUS_MASK:
        print("  warning: error status bits are set")

    profile = read_profile_counters()
    print_profile_counters(profile)
    pingpong = summarize_pingpong(profile)

    got = read_outputs()
    mismatches = compare_outputs(got, cfg["golden"][:GOLDEN_WORDS])
    elapsed = time.time() - t0

    passed = len(mismatches) == 0 and (status & ERROR_STATUS_MASK) == 0
    print(f"case{case_id}: {'PASS' if passed else 'FAIL'} ({elapsed:.2f}s, mismatches={len(mismatches)})")
    return {
        "case": case_id,
        "passed": passed,
        "mismatches": mismatches,
        "status": status,
        "elapsed_s": elapsed,
        "outputs": got,
        "profile": profile,
        "pingpong": pingpong,
    }

In [7]:
# Quick smoke test: run only case1 first.
# If this passes, run the all-case cell below.
result_case1 = run_case(1, reload_before=True, verbose=True)
result_case1["passed"]


=== Running case1 ===
k_cnt=1, data_words=64, total_tiles=1
  activation: done, 0x00000005 [module_ready, input_done]
  bias: done, 0x00000005 [module_ready, input_done]
  weight tile 0: done, 0x00000085 [module_ready, input_done, weight_active_valid]
  final status: 0x000000A5 [module_ready, input_done, output_done, weight_active_valid]
  hardware counters:
    total_cycle_count: 516
    compute_busy_cycles: 195
    weight_load_cycles: 0
    overlap_cycles: 0
    stall_cycles_due_to_w_bram_valid_0: 0
    systolic_load_cycles: 65
    weight_word_accepts: 0
    activation_bram_reads: 64
    weight_bram_reads: 64
    bias_bram_reads: 16
    output_word_writes: 256
    input_bram_access_cycles: 64
    core_bram_access_cycles: 320
    no_overlap_total_cycles_est: 516 (actual total + actual overlap)
    pingpong_speedup_vs_no_overlap_est: 1.0000x
case1: PASS (0.03s, mismatches=0)


True

In [8]:
# Run case1~case4. Case4 writes 6144 activation words and 96 weight tiles, so it takes longer.
results = []
for case_id in [1, 2, 3, 4]:
    results.append(run_case(case_id, reload_before=True, verbose=(case_id != 4)))

print("\n=== Summary ===")
for r in results:
    p = r["profile"]
    pp = r["pingpong"]
    print(
        f"case{r['case']}: {'PASS' if r['passed'] else 'FAIL'}, "
        f"elapsed={r['elapsed_s']:.2f}s, mismatches={len(r['mismatches'])}, "
        f"hw_cycles={p['total_cycle_count']}, overlap={p['overlap_cycles']}, "
        f"stall_w_valid0={p['stall_cycles_due_to_w_bram_valid_0']}, "
        f"no_overlap_est={pp['no_overlap_total_cycles_est']}, "
        f"speedup_est={pp['pingpong_speedup_vs_no_overlap_est']:.4f}x, "
        f"status={format_status(r['status'])}"
    )

all_passed = all(r["passed"] for r in results)
print("ALL PASS" if all_passed else "SOME CASES FAILED")


=== Running case1 ===
k_cnt=1, data_words=64, total_tiles=1
  activation: done, 0x00000005 [module_ready, input_done]
  bias: done, 0x00000005 [module_ready, input_done]
  weight tile 0: done, 0x00000085 [module_ready, input_done, weight_active_valid]
  final status: 0x000000A5 [module_ready, input_done, output_done, weight_active_valid]
  hardware counters:
    total_cycle_count: 516
    compute_busy_cycles: 195
    weight_load_cycles: 0
    overlap_cycles: 0
    stall_cycles_due_to_w_bram_valid_0: 0
    systolic_load_cycles: 65
    weight_word_accepts: 0
    activation_bram_reads: 64
    weight_bram_reads: 64
    bias_bram_reads: 16
    output_word_writes: 256
    input_bram_access_cycles: 64
    core_bram_access_cycles: 320
    no_overlap_total_cycles_est: 516 (actual total + actual overlap)
    pingpong_speedup_vs_no_overlap_est: 1.0000x
case1: PASS (0.02s, mismatches=0)

=== Running case2 ===
k_cnt=2, data_words=128, total_tiles=2
  activation: done, 0x00000005 [module_ready, inp


## Hardware Counter Profiling and Modeled Traffic

The cycle, BRAM access, and ping-pong numbers below are read from RTL counters in `SystolicAxiLiteWrapper`, so they are hardware-measured for this AXI-Lite correctness design.

The DRAM traffic numbers are still a correctness/profiling model, not actual DDR/DMA measurements. In this model, every 32-bit Python word written into `InputLoadFSM` is counted as:

- 1 modeled DRAM read
- 1 modeled BRAM write

This matches the intended final architecture at a high level:

`DRAM -> tile loader -> on-chip BRAM -> systolic -> output buffer`

For no-overlap comparison, the notebook estimates `without ping-pong / no overlap` as:

`no_overlap_total_cycles_est = actual_total_cycle_count + actual_overlap_cycles`

So the with-ping-pong side is measured by hardware, while the without-ping-pong side is a simple serialized baseline estimate.

Energy numbers need power inputs from Vivado `report_power` or from a chosen memory-energy model. They are computed from RTL-measured cycles and access counts, not from Python elapsed time.

In [9]:
FREQ_MHZ = 100.0
WORD_BYTES = 4

# Filled from reports/power_impl_vectorless.rpt.
# These are Vivado vectorless implementation estimates, not case-input activity.
#
# Suggested Vivado source:
#   open_run impl_1
#   report_power -file reports/power_vectorless_impl.rpt
#
# For SAIF-based per-case power, use the case report_power flow instead.
VIVADO_CORE_DYNAMIC_POWER_W = 0.137    # SystolicAxiLiteWrapp_0 hierarchy dynamic power
VIVADO_BRAM_DYNAMIC_POWER_W = 0.001    # Upper bound; report shows Block RAM dynamic power <0.001 W
VIVADO_LEAKAGE_POWER_W = 0.145         # Device Static Power

# This design does not perform real DDR transactions yet. Set this only if you have
# a DDR datasheet/model number for one 32-bit modeled DRAM access.
DRAM_ENERGY_PER_32B_ACCESS_J = None

# Modeled DRAM loader bandwidth. 1 means one 32-bit word per cycle.
DRAM_WORDS_PER_CYCLE_MODEL = 1

def mac_count_for_case(case_id):
    return 16 * 16 * 16 * CASE_CONFIG[case_id]["k_cnt"]

def div0(num, den):
    return (num / den) if den else 0.0

def ceil_div(num, den):
    return (int(num) + int(den) - 1) // int(den)

def maybe_energy_per(value_w, cycles, count):
    if value_w is None or count == 0:
        return None
    active_time_s = cycles / (FREQ_MHZ * 1e6)
    return (value_w * active_time_s) / count

def fmt_metric(value, digits=4):
    if value is None:
        return "N/A"
    if isinstance(value, float):
        return f"{value:.{digits}e}" if abs(value) < 1e-3 and value != 0 else f"{value:.{digits}f}"
    return str(value)

def modeled_traffic_for_result(result, fused_output=False, next_op_reads_intermediate=True):
    case_id = result["case"]
    cfg = CASE_CONFIG[case_id]
    profile = result["profile"]

    act_words = cfg["data_words"]
    weight_words = cfg["data_words"]
    bias_words = BIAS_WORDS
    output_words = int(profile["output_word_writes"])

    modeled_input_dram_reads = act_words + weight_words + bias_words
    modeled_input_bram_writes = modeled_input_dram_reads

    output_dram_writes = 0 if fused_output else output_words
    fusion_saved_output_writes = output_words if fused_output else 0
    fusion_saved_next_reads = output_words if (fused_output and next_op_reads_intermediate) else 0
    fusion_saved_total_words = fusion_saved_output_writes + fusion_saved_next_reads

    modeled_dram_words = modeled_input_dram_reads + output_dram_writes
    modeled_dram_access_time_cycles = ceil_div(modeled_dram_words, DRAM_WORDS_PER_CYCLE_MODEL)
    modeled_dram_bandwidth_bytes_per_cycle = div0(
        modeled_dram_words * WORD_BYTES,
        modeled_dram_access_time_cycles,
    )

    actual_input_bram_read_words = (
        int(profile["activation_bram_reads"])
        + int(profile["weight_bram_reads"])
        + int(profile["bias_bram_reads"])
    )
    actual_core_bram_access_words = actual_input_bram_read_words + output_words
    actual_input_bram_access_cycles = int(profile["input_bram_access_cycles"])
    actual_core_bram_access_cycles = int(profile["core_bram_access_cycles"])
    served_input_bram_bandwidth_bytes_per_cycle = div0(
        actual_input_bram_read_words * WORD_BYTES,
        actual_input_bram_access_cycles,
    )
    effective_input_bram_bandwidth_bytes_per_cycle = div0(
        actual_input_bram_read_words * WORD_BYTES,
        int(profile["systolic_load_cycles"]),
    )
    actual_core_bram_bandwidth_bytes_per_cycle = div0(
        actual_core_bram_access_words * WORD_BYTES,
        actual_core_bram_access_cycles,
    )

    total_cycles = int(profile["total_cycle_count"])
    macs = mac_count_for_case(case_id)
    energy_per_mac = maybe_energy_per(VIVADO_CORE_DYNAMIC_POWER_W, total_cycles, macs)
    energy_per_bram_access = maybe_energy_per(
        VIVADO_BRAM_DYNAMIC_POWER_W,
        total_cycles,
        actual_core_bram_access_words,
    )
    energy_per_dram_access = DRAM_ENERGY_PER_32B_ACCESS_J

    return {
        "case": case_id,
        "k_cnt": cfg["k_cnt"],
        "MACs": macs,
        "modeled_act_DRAM_reads_w": act_words,
        "modeled_weight_DRAM_reads_w": weight_words,
        "modeled_bias_DRAM_reads_w": bias_words,
        "modeled_input_BRAM_writes_w": modeled_input_bram_writes,
        "actual_act_BRAM_reads_w": int(profile["activation_bram_reads"]),
        "actual_weight_BRAM_reads_w": int(profile["weight_bram_reads"]),
        "actual_bias_BRAM_reads_w": int(profile["bias_bram_reads"]),
        "actual_output_BRAM_writes_w": output_words,
        "actual_input_BRAM_access_time_cycles": actual_input_bram_access_cycles,
        "actual_core_BRAM_access_time_cycles": actual_core_bram_access_cycles,
        "served_input_BRAM_bandwidth_B_per_cycle": served_input_bram_bandwidth_bytes_per_cycle,
        "effective_input_BRAM_bandwidth_B_per_cycle": effective_input_bram_bandwidth_bytes_per_cycle,
        "actual_input_BRAM_bandwidth_B_per_cycle": served_input_bram_bandwidth_bytes_per_cycle,
        "actual_core_BRAM_bandwidth_B_per_cycle": actual_core_bram_bandwidth_bytes_per_cycle,
        "modeled_output_DRAM_writes_w": output_dram_writes,
        "modeled_DRAM_access_time_cycles": modeled_dram_access_time_cycles,
        "modeled_DRAM_bandwidth_B_per_cycle": modeled_dram_bandwidth_bytes_per_cycle,
        "fusion_saved_output_writes_w": fusion_saved_output_writes,
        "fusion_saved_next_reads_w": fusion_saved_next_reads,
        "fusion_saved_total_words": fusion_saved_total_words,
        "modeled_DRAM_words": modeled_dram_words,
        "modeled_DRAM_bytes": modeled_dram_words * WORD_BYTES,
        "actual_total_cycles": total_cycles,
        "actual_compute_busy_cycles": int(profile["compute_busy_cycles"]),
        "actual_systolic_load_cycles": int(profile["systolic_load_cycles"]),
        "actual_weight_load_cycles": int(profile["weight_load_cycles"]),
        "actual_overlap_cycles": int(profile["overlap_cycles"]),
        "actual_stall_w_valid0_cycles": int(profile["stall_cycles_due_to_w_bram_valid_0"]),
        "no_overlap_total_cycles_est": int(result["pingpong"]["no_overlap_total_cycles_est"]),
        "pingpong_saved_cycles_actual": int(result["pingpong"]["pingpong_saved_cycles_actual"]),
        "pingpong_speedup_vs_no_overlap_est": result["pingpong"]["pingpong_speedup_vs_no_overlap_est"],
        "latency_us_from_actual_cycles": total_cycles / FREQ_MHZ if total_cycles else 0.0,
        "GOPS_from_actual_cycles": (macs / total_cycles) * FREQ_MHZ / 1000.0 if total_cycles else 0.0,
        "MACs_per_modeled_DRAM_byte": macs / (modeled_dram_words * WORD_BYTES) if modeled_dram_words else 0.0,
        "Energy_per_MAC_J": energy_per_mac,
        "Energy_per_BRAM_access_J": energy_per_bram_access,
        "Energy_per_DRAM_access_J": energy_per_dram_access,
        "Leakage_Power_W": VIVADO_LEAKAGE_POWER_W,
    }

def print_table(rows, columns):
    widths = {}
    for c in columns:
        widths[c] = max(
            len(c),
            *(len(fmt_metric(r[c])) for r in rows),
        )
    print(" | ".join(c.ljust(widths[c]) for c in columns))
    print("-+-".join("-" * widths[c] for c in columns))
    for r in rows:
        vals = []
        for c in columns:
            v = r[c]
            vals.append(fmt_metric(v).ljust(widths[c]))
        print(" | ".join(vals))

if "results" not in globals():
    raise RuntimeError("Run the case1~case4 cell first so hardware counters are available in results.")

profiles_no_fusion = [modeled_traffic_for_result(r, fused_output=False) for r in results]
profiles_fused = [modeled_traffic_for_result(r, fused_output=True) for r in results]

hardware_columns = [
    "case", "k_cnt", "MACs",
    "actual_total_cycles", "actual_compute_busy_cycles",
    "actual_systolic_load_cycles", "actual_weight_load_cycles",
    "actual_overlap_cycles", "actual_stall_w_valid0_cycles",
    "no_overlap_total_cycles_est", "pingpong_saved_cycles_actual",
    "pingpong_speedup_vs_no_overlap_est",
    "latency_us_from_actual_cycles", "GOPS_from_actual_cycles",
]

traffic_columns = [
    "case",
    "modeled_act_DRAM_reads_w", "actual_act_BRAM_reads_w",
    "modeled_weight_DRAM_reads_w", "actual_weight_BRAM_reads_w",
    "modeled_bias_DRAM_reads_w", "actual_bias_BRAM_reads_w",
    "modeled_input_BRAM_writes_w", "actual_output_BRAM_writes_w",
    "actual_input_BRAM_access_time_cycles",
    "served_input_BRAM_bandwidth_B_per_cycle",
    "effective_input_BRAM_bandwidth_B_per_cycle",
    "actual_core_BRAM_access_time_cycles",
    "actual_core_BRAM_bandwidth_B_per_cycle",
    "modeled_output_DRAM_writes_w",
    "modeled_DRAM_access_time_cycles",
    "modeled_DRAM_bandwidth_B_per_cycle",
    "fusion_saved_output_writes_w", "fusion_saved_next_reads_w",
    "fusion_saved_total_words", "modeled_DRAM_bytes",
    "MACs_per_modeled_DRAM_byte",
]

energy_columns = [
    "case",
    "Energy_per_MAC_J",
    "Energy_per_BRAM_access_J",
    "Energy_per_DRAM_access_J",
    "Leakage_Power_W",
]

print("Hardware-measured cycle / ping-pong counters:")
print_table(profiles_no_fusion, hardware_columns)

print("\nModeled traffic, no operator fusion:")
print_table(profiles_no_fusion, traffic_columns)

print("\nModeled traffic, fused output kept on chip:")
print_table(profiles_fused, traffic_columns)

print("\nEnergy metrics:")
print_table(profiles_no_fusion, energy_columns)
print(
    "\nEnergy note: N/A means the corresponding Vivado/report/model power input "
    "above is still None."
)

Hardware-measured cycle / ping-pong counters:
case | k_cnt | MACs   | actual_total_cycles | actual_compute_busy_cycles | actual_systolic_load_cycles | actual_weight_load_cycles | actual_overlap_cycles | actual_stall_w_valid0_cycles | no_overlap_total_cycles_est | pingpong_saved_cycles_actual | pingpong_speedup_vs_no_overlap_est | latency_us_from_actual_cycles | GOPS_from_actual_cycles
-----+-------+--------+---------------------+----------------------------+-----------------------------+---------------------------+-----------------------+------------------------------+-----------------------------+------------------------------+------------------------------------+-------------------------------+------------------------
1    | 1     | 4096   | 516                 | 195                        | 65                          | 0                         | 0                     | 0                            | 516                         | 0                            | 1.0000               

## Notes

- `case1` is the first smoke test.
- `case2` to `case4` use multiple weight tiles. The notebook follows the RTL testbench flow: load activation, load bias, load weight tile 0, start the systolic core, then load remaining weight tiles while the core runs.
- `total_cycle_count`, `compute_busy_cycles`, `weight_load_cycles`, `overlap_cycles`, `stall_cycles_due_to_w_bram_valid_0`, BRAM reads, output writes, and BRAM access-time cycles come from hardware counters.
- `served_input_BRAM_bandwidth_B_per_cycle = input_BRAM_read_bytes / input_bram_access_cycles`; this measures bandwidth only when BRAM actually serves a read.
- `effective_input_BRAM_bandwidth_B_per_cycle = input_BRAM_read_bytes / systolic_load_cycles`; this includes load-state waiting/stall and therefore reflects Python/MMIO or future DRAM-loader readiness.
- `modeled_*DRAM*` values are not actual DDR measurements. They treat Python/MMIO input writes as a proxy for future DRAM -> tile loader -> BRAM movement.
- Actual DDR bandwidth and DMA timing should be measured later after replacing this correctness path with AXI DMA or an AXI master connected to the Zynq HP port.
- `Energy_per_MAC_J` and `Energy_per_BRAM_access_J` are computed from Vivado power numbers that you enter in the profiling cell; they are not RTL counters.
- `Energy_per_DRAM_access_J` must come from an external DDR energy model/datasheet because the current RTL does not issue actual DDR transactions.